In [1]:
#from tensorflow import keras
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd

# IMPORTAMOS LOS DATOS
init_df = pd.read_csv("ipc.csv", skiprows=1) #, usecols=lambda column: column >= 2
init_df.head()



2023-10-16 21:46:26.932820: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-10-16 21:46:26.936091: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 21:46:26.981128: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 21:46:26.983156: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-16 21:46:27.829173: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

,1989-05-01,603.0082,608.7225,612.8195,589.7707,607.9455,540.3457,679.1996,693.9204,525.8695,...,346.3501,346.3501.1,750.0142,757.2518,783.5692,649.7888,402.1424,402.1424.1,454.2819,454.2819.1
0,1989-06-01,1293.2936,1395.7573,1332.7712,1306.3422,1348.5443,1153.9080,1319.8834,1381.6045,1072.8707,...,926.4865,926.4865,1718.6736,1745.5862,1725.7064,1578.0661,697.5486,697.5486,705.1612,705.1612
1,1989-07-01,3836.3442,4033.9027,4236.6526,4537.4227,3865.6753,3796.3507,4438.1639,4575.4766,3040.3962,...,2351.4228,2351.4228,5162.3131,5445.0025,4889.7094,4368.3077,1937.7715,1937.7715,1900.5661,1900.5661
2,1989-08-01,5288.8345,5029.6921,4774.7054,4680.3308,5174.5838,4191.1678,5110.5971,4848.1587,4203.3324,...,6960.2114,6960.2114,6829.9162,7038.9692,6429.4920,6634.6483,3150.6250,3150.6250,4666.0843,4666.0843
3,1989-09-01,5783.6203,5499.6552,4729.6412,4473.2784,5113.9576,4237.2707,5014.6478,4771.7241,4662.8708,...,7677.1132,7677.1132,6807.1577,7170.4527,5856.4938,6970.0562,4047.7432,4047.7432,8657.0679,8657.0679
4,1989-10-01,6107.2335,5748.7661,4749.1084,4387.0599,5113.6002,4256.3364,4953.3701,4767.3491,5086.6990,...,7991.8748,7991.8748,6823.6918,7196.2246,5751.5854,7182.4586,4880.9909,4880.9909,9315.4223,9315.4223


In [3]:
dataset = init_df.iloc[:, 2 : 11]
dataset.head()

,608.7225,612.8195,589.7707,607.9455,540.3457,679.1996,693.9204,525.8695,596.277
0,1395.7573,1332.7712,1306.3422,1348.5443,1153.9080,1319.8834,1381.6045,1072.8707,1642.9623
1,4033.9027,4236.6526,4537.4227,3865.6753,3796.3507,4438.1639,4575.4766,3040.3962,5122.1088
2,5029.6921,4774.7054,4680.3308,5174.5838,4191.1678,5110.5971,4848.1587,4203.3324,5762.0238
3,5499.6552,4729.6412,4473.2784,5113.9576,4237.2707,5014.6478,4771.7241,4662.8708,5756.2825
4,5748.7661,4749.1084,4387.0599,5113.6002,4256.3364,4953.3701,4767.3491,5086.6990,5836.8440


In [10]:
# CREAMOS REFERENCIA A UN NORMALIZAR (MAXIMO MINIMO EN ESTE CASO)
scaler = MinMaxScaler()
# NORMALIZAMOS LOS DATOS
scaled_dataset = scaler.fit_transform(dataset)

# DEFINIMOS LA CANTIDAD DE DATOS PARA EL ENTRENAMIENTO
train_set_size = 100

# GUARDAMOS EL TAMAÑO TOTAL DEL DATASET
set_size = len(scaled_dataset)

# DIVIDIMOS LOS DATOS PARA ENTRENAMIENTO
dataset_train = scaled_dataset[0:train_set_size]

# DIVIDIMOS LOS DATOS PARA COMPROBACIÓN
dataset_test = scaled_dataset[train_set_size:set_size]

# EXTRAEMOS LAS VARIABLES EXPLICATIVAS PARA LA COMPROBACIÓN
vars_explicativas_test = dataset_test[:, 0:8]

# EXTRAEMOS LAS VARIABLES EXPLICADAS PARA LA COMPROBACIÓN
var_explicada_test = dataset_test[:, 1]

# EXTRAEMOS LAS VARIABLES EXPLICATIVAS PARA EL ENTRENAMIENTO
X = dataset_train[:, 0:8]

# EXTRAEMOS LAS VARIABLES EXPLICADAS PARA EL ENTRENAMIENTO
Y = dataset_train[:, 1]
Y = Y[:, np.newaxis]



In [11]:
# CREAMOS REFERENCIA A UN MODELO RNA SECUENCIAL (SE DEFINE CAPA POR CAPA)
model = Sequential()

# CREAMOS LA TOPOLOGIA DE LA RNA CON SUS DISTINTOS PARÁMETROS
model.add(Dense(9, input_dim=8, activation='relu'))
model.add(Dense(150, activation='relu'))
model.add(Dense(500, activation='relu'))
model.add(Dense(150, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# CREAMOS UNA REFERENCIA A UN ALG. DE OPTIMIZACIÓN (SGD) CON SU TASA DE APRENDIZAJE
opt = 'sgd'
#opt = keras.optimizers.SGD(learning_rate=0.05)

# COMPILAMOS EL MODELO CON LA FUNCIÓN ECM DE EVALUACIÓN DE ERROR
model.compile(loss='mean_squared_error', optimizer=opt)

# ENTRENAMOS EL MODELO DANDOLE LAS VARIABLES Y DEMÁS PARÁMETROS
model.fit(X, Y, epochs=1000, batch_size=10, verbose=0)

# EVALUA EL MODELO
scores = model.evaluate(X, Y)

# DADO EL MODELO ENTRENADO Y LAS VARIABLES QUE GUARDAMOS ANTERIORMENTE, PREDECIMOS
predicciones = model.predict(vars_explicativas_test)[:, 0]

# DEFINIMOS LA FUNCIÓN DE ERROR ECM
ecm_f = lambda Yp, Yr: np.mean((Yp - Yr) ** 2)

# DADA LAS PREDICCIONES Y LOS VALORES REALES, EVALUAMOS EL ERROR REAL DEL MODELO
ecm_real = ecm_f(predicciones, var_explicada_test)

# CREAMOS UNA TABLA PARA IMPRIMIR
d = {'predicciones': predicciones, 'reales': var_explicada_test}

# IMPRIMIMOS LA TABLA
#display(pd.DataFrame(d))
print(pd.DataFrame(d))


# IMPRIMIMOS PARÁMETROS DE ERROR REAL
print("ECM Real: " + str(ecm_real))
c = np.corrcoef(predicciones, var_explicada_test)
print("Correlación de Pearson: " + str(c))

2/2 [==============================] - 0s 4ms/step
    predicciones    reales
0       0.918552  0.952963
1       0.917718  0.950598
2       0.918314  0.951665
3       0.917125  0.946662
4       0.917240  0.945080
5       0.917511  0.945730
6       0.917540  0.943180
7       0.917803  0.944607
8       0.917614  0.946780
9       0.917553  0.947278
10      0.917707  0.947073
11      0.917057  0.945033
12      0.916569  0.944682
13      0.916389  0.944971
14      0.915672  0.943399
15      0.914831  0.942114
16      0.913837  0.940138
17      0.913415  0.937554
18      0.912506  0.932480
19      0.911913  0.930935
20      0.911000  0.928619
21      0.911044  0.928955
22      0.911010  0.928294
23      0.909897  0.925038
24      0.909361  0.924002
25      0.904922  0.917258
26      0.901760  0.913353
27      0.903561  0.911526
28      0.903764  0.910132
29      0.904191  0.910248
30      0.903415  0.907976
31      0.902818  0.907150
32      0.902045  0.905218
33      0.900924  0.903018
34  